In [ ]:
### ResNet18 + CIFAR10 학습 / 추론 / 시각화
    - CIFAR10 데이터 로드 및 전처리
    - ResNet18 구성
    - ResNet18 튜닝
    - 학습 및 추론
    - Confusion Matrix, Activation Map, Conv Kernel 시각화

In [ ]:
### CIFAR10 데이터 로드 및 전처리
    - 10 개의 class, 32 x 32 x 3 채널 이미지
    - CNN 입력은 C x H x W 이므로 변환 필요

# CIFAR-10에서 널리 쓰는 normalize 값
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)  # 채널별 평균(R,G,B)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)  # 채널별 표준편차(R,G,B)

# 데이터 전처리 transforms
train_tfms = T.Compose([
    T.RandomCrop( 32, padding=4 ),              # 4픽셀 패딩 후 32x32 랜덤 크롭
    T.RandomHorizontalFlip(),                   # 좌우 랜덤 반전
    T.ToTensor(),                               # PIL -> torch tensor (C, H, W)
    T.Normalize( CIFAR10_MEAN, CIFAR10_STD )    # 정규화, -2.5 ~ 2.5
])

test_tfms = T.Compose([
    T.ToTensor(),                               # PIL -> torch tensor (C, H, W)
    T.Normalize( CIFAR10_MEAN, CIFAR10_STD )    # 정규화, -2.5 ~ 2.5
])

# 데이터 로드
data_root = "./data"
train_set = torchvision.dataset.CIFAR10( data_root, train=True, download=True, transform=train_tfms )
test_set = torchvision.dataset.CIFAR10( data_root, train=False, download=True, transform=test_tfms )

class_names = train_set.classes
num_classes = 10
batch_size = 256
num_workers = min(8, os.cpu_count() or 2)

train_loader = DataLoader( train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True )
test_loader = DataLoader( test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True )

In [ ]:
### denorm(x) 이미지로 역변환하는 유틸리티 함수

def denorm( x ):
    mean = torch.tensor( CIFAR10_MEAN ).view( 1, 3, 1, 1 )
    std = torch.tensor( CIFAR10_STD ).view( 1, 3, 1, 1 )
    return x * std + mean   # 정규화 역변환, 0 ~ 1

## 테스트
images, labels = next(itr(train_loader))
image_dn = denorm(image[:16]).clamp(0, 1)

grid = torchvision.utils.make_grid(images_dn, nrow=8)
plt.figure( figsize=(12, 4) )
plt.imshow( grid.permute(1, 2, 0) ) # CHW -> HWC
plt.axis( 'off' )
plt.title( 'CIFAR-10 samples (denormalized)' )
plt.show()

In [ ]:
### ResNet18 모델 로드 및 CIFAR10 에 맞게 튜닝

def build_resnet18_for_cifar10( num_classes = 10 ) :
    model = models.resnet18( weights=None )     # pretrained 없이 로드

    # CIFAR10 에 맞게 튜닝
    model.conv1 = nn.Conv2d( 3, 64, kernel_size=3, stride=1, padding=1, bias=False )    # RGB 3채널 -> 64채널
    model.maxpool = nn.Identity()                                                       # max pool 제거
    model.fc = nn.Linear( model.fc.in_features, num_classes )                           # 최종 10개 분류
    
    return model

model = build_resnet18_for_cifar10( num_classes ).to(device)
num_params = sum( p.numel() for p in model.parameters() )

In [ ]:
### ResNet18 모델 학습 / 평가 용 유틸리티 함수
    - TrainConfig class
    - accuracy_top1()
    - train_one_epoch()
    - evaluate()

@dataclass
class TrainConfig:
    base_epochs_if_new: int = 20
    extra_epochs_if_resume: int = 10
    lr: flaot = 0.1
    weight_decay: float = 5e-4
    momentum: float = 0.9
    label_sommthing: float = 0.0

cfg = TrainConfig()


## 정확도 계산 함수
def accuracy_top1( logits, targets ) :
    preds = logits.argmax( dim=1 )
    return (preds == targets).float().mean().item()


## 학습 함수
scaler = torch.amp.GradScaler( enabled=torch.cuda.is_available() )

def train_one_epoch( model, loader, optimizer, criterion ):
    model.train()   # 학습 모드
    total_loss, total_acc = 0.0, 0.0
    
    t0 = time.time()
    n = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)   # asyncronous transfer
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast( device_type=device.type, enabled=torch.cuda.is_available() ) : # AMP autocast
            logits = model( images )
            loss = criterion( logits, labels )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = images.size()
        total_loss += loss.item() * bs
        total_acc += accuracy_top1( logits.detach(), labels ) * bs
        n += bs
    
    dt = time.time() - t0
    return total_loss / n, total_acc / n, dt    # 평균 loss, acc, 시간


## 평가 함수
@torch.no_grad()
def evaluate( model, loader, criterion ) :
    model.eval()    # 평가 모드
    total_loss, total_acc = 0.0, 0.0

    n = 0
    for images, labels in loader :
        images = images.to(device, non_blocking=True)   # asyncronous transfer
        labels = labels.to(device, non_blocking=True)

        logits = model( images )
        loss = criterion( logits, labels )

        bs = images.size(0)
        total_loss += loss.item() * bs
        total_acc = accuracy_top1( logtis, labels ) * bs
        n += bs

    return total_loss / n, total_acc / n

In [ ]:
### ResNet18 학습

## 1. loss function / optimizer 생성
criterion = nn.CrossEntropyLoss( label_smoothing=cfg.label_smoothing )
optimizer = torch.optim.SGD( model.parameters(), lr=cfg.lr, momentum=cfg.momentum, weight_decay=cfg.weight_decay )  # SGD Optimizer

start_epoch = 0
best_acc = -1.0

## 2. 모델 로드
ckpt_path = 'resnet18_cifar10.pth'  # best checkpoint 경로
ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
model.load_state_dic( ckpt['model'] )
start_epoch = int(ckpt.get("epoch", 0))
best_acc = float(ckpt.get("val_acc", -1.0))
epochs_to_run = cfg.extra_epochs_if_resume

## 3. 모델 학습
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}  # 기록용 dict

scheduler = torch.optim.lr_scheduler.CosinAnnealingLR( optimizer, T_max=epochs_to_run ) ## 학습률을 곡선으로 변화

for e in range(1, epochs_to_run+1):
    epoch = start_epoch + e
    lr_now = optimizer.param_groups[0]['lr']
    tr_loss, tr_acc, dt = train_one_epoch( model, train_loader, optimizer, criterion )
    va_loss, va_acc = evaluate( model, test_loader, criterion )
    scheduler.step()    # lr 업데이트

    history['train_loss'].append(tr_loss)  # 기록
    history['train_acc'].append(tr_acc)  # 기록
    history['val_loss'].append(va_loss)  # 기록
    history['val_acc'].append(va_acc)  # 기록
    history['lr'].append(lr_now)  # 기록

    print(f"Epoch {epoch:03d} | lr {lr_now:.4f} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {va_loss:.4f} acc {va_acc:.4f} | {dt:.1f}s")

    if va_acc > best_acc :
        best_acc = va_acc
        torch.save( { 'model': model.state_dict(), 'epoch': epoch, 'val_acc': best_acc, 'config': cofig.__dict__}, ckpt_path )  # 체크포인트 저장

In [ ]:
### ResNet18 추론

ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)  # best checkpoint 로드
model.load_state_dict(ckpt['model'])  # 최신 모델 가중치 적용
model.eval()  # eval 모드, i.e., Dropout 비활성화, BN Running Stats 고정
print('Loaded checkpoint epoch:', ckpt.get('epoch'), 'val_acc:', ckpt.get('val_acc'))  # 정보 출력

@torch.no_grad()  # 자동 미분(Autograd) 기능 비활성화
def predict_batch( model, images ):
    logits = model( images )  # raw logits
    probs = F.softmax( logits, dim=1 )  # 확률로 변환
    conf, pred = probs.max( dim=1 )  # 최대 확률(conf)과 클래스(pred)
    return pred, conf, probs

images, labels = next(iter(test_loader))  # test 배치 하나
images = images.to(device)  # GPU 이동
labels = labels.to(device)  # GPU 이동
pred, conf, probs = predict_batch(model, images)  # 예측

k = 16  # 시각화 샘플 수
imgs_dn = denorm(images[:k].cpu()).clamp(0,1)  # 역정규화

grid = torchvision.utils.make_grid(imgs_dn, nrow=8)  # 그리드
plt.figure(figsize=(12,6))
plt.imshow(grid.permute(1,2,0))  # HWC로 변환
plt.axis('off')
plt.title('Inference samples')
plt.show()

for i in range(k):
    gt = class_names[int(labels[i])]  # 정답 클래스명
    pr = class_names[int(pred[i])]  # 예측 클래스명
    cf = float(conf[i])  # confidence
    print(f"[{i:02d}] GT={gt:>10s} | Pred={pr:>10s} | conf={cf:.3f}")  # 출력

In [ ]:
### Confusion Matrix 로 클래스 별 정확도 및 오분류 샘플 체크

@torch.no_grad()
def collect_predictions( model, loader ):
    model.eval()
    all_preds, all_labels, all_images = [], [], []
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        preds, conf, _ = predict_batch( model, images )

        all_preds.append( preds.cpu() )
        all_labels.append( labels.cpu() )
        all_images.append( images.cpu() )

    return torch.cat(all_images), torch.cat(all_preds), torch.cat(all_labels)   # tensor 로 결합


def confusion_and_perclass( preds, labels, num_classes ) :
    cm = torch.zeros( (num_classes, num_classes), dtype=torch.int64 )   # confusion matrix 초기화

    for t, p in zip( labels, preds ) :
        cm[ int(t), int(p) ] += 1   # GT = t, Pred = p 해당 인덱스의 카운트 증가
    
    per_class_acc = cm.diagonal().float() / torch.clamp( cm.sum(dim=1).float, min=1.0 ) # 클래스 별 정확도
    return cm, per_class_acc


images_all, preds_all, labels_all = collect_predictions( model, test_loader )
cm, pacc = confusion_and_perclass( preds_all, labels_all, num_classes )

# show cm
# ...

print( "Per-class accurarcy:" )
for i, a in enumerate(pacc.tolist()):
    print( f"{class_names[i]:>10s}: {a:.3f}" )


## 오분류 샘플 16개 이미지로 확인 
mis_idx = (preds_all != labels_all).nonzero( as_tuple=False ).squeeze(1)    # 오분류 인덱스. squeeze(1) 1번째 인덱스 차원 제거

if mis_idx.numel() > 0 :
    sel = mis_idx[ torch.randperm( mis_idx.numel() )[:16] ] # 랜덤 16개 추출
    imgs = denorm( images_all[ sel ] ).clamp(0, 1)
    
    grid = torchvision.utils.make_grid(imgs, nrow=8)    # 그리드 2 x 8
    ... # plt.show

    for j, idx in enumerate( sel.tolist() ):
        gt = class_names[ int(labels_all[idx]) ]    # GT class name
        pr = class_names[ int(preds_all[idx]) ]     # Pred class name
        print( f"[{j:02d}] idx={idx:5d} GT={gt:>10s} Pred={pr:>10s}" )


In [ ]:
### Activation Map 시각화 (채널별)
    - 레이어에 register_forward_hook() 을 걸어서 feature_map(B, C, H, W) 저장

def get_module_by_name( model: nn.Module, name: str ) -> nn.Module :
    cur = model
    for part in name.split( "." ):
        if part.isdigit():
            cur = cur[ int(part) ]      # sequential 인덱스 접근
        else:
            cur = getattr( cur, part )  # attribute 접근

    return cur


def capture_activation( model, layer_name, x_single ) :
    layer = get_module_by_name( model, layer_name )
    activ = {}

    ## callback hook function
    def hook( m, i, o ) :
        activ[ 'feat' ] = o.detach().cpu()  # (B, C, H, W) 저장

    h = layer.register_forward_hook( hook )

    model.eval()
    with torch.no_grad():
        _ = model( x_single )   # forward 수행

    h.remove()

    return activ[ 'feat' ][ 0 ] # (C, H, W) 리턴


def show_topk_activation_grid( feat_chw, title, topk=16 ) :
    ch_scores = feat_chw.abs().maan(dim=(1,2))  # 채널 별 평균 크기
    idx = torch.topk( ch_scores, k=min(topk, feat_chw.shape[0]) ).indices   # top k 채널 인덱스

    maps = []   # (K, 1, H, W) 로 만들 결과 리스트
    for ci in idx :
        m = feat_chw[ int(ci) ]     # (H, W)
        m = (m - m.min()) / (m.max() - m.min() + 1e-6)  # 0 ~ 1 정규화
        maps.append( m[ None, None, ... ] )     # (1, 1, H, W)
    maps = torch.cat(maps, dim=0)   # (K, 1, H, W)

    grid = torchvision.utils.make_grid(maps, nrows=8, padding=2)
    plt.figure(figsize=(12,5))
    if grid.ndim == 3 and grid.shape[0] in (3, 4):  # (3, H, W) 라면 HWC 로 변환
        plt.imshow( grid.permute(1, 2, 0) )
    else:
        plt.imshow( grid.squeeze(0), cmap='viridis' )
    plt.title(title)
    plt.axis('off')
    plt.show()


## 1. 시각화 할 레이어 이름 정의
target_layers = {
    "Layer 1": "layer1.1.conv2d",
    "Layer 2": "layer2.1.conv2d",
    "Layer 3": "layer3.1.conv2d",
    "Layer 4": "layer4.1.conv2d"
}

## 2. 입력 이미지 선택
idx = 10 # 샘플 인덱스
x, y = test_set[ idx ]
x_in = x.unsqueeze(0).to(device)

img0 = denorm( x.unsqueeze(0) ).squeeze(0).clamp(0, 1)
# ... plt.imshow( img0.permute(1, 2, 0) )

## 3. 반복문을 통해 각 레이어의 Activation 추출 및 시각화
for title, layer_name in target_layers.item():
    # Activation 캡쳐
    feat = capture_activation( model, layer_name, x_in )

    # 정보 출력
    print( f"{layer_name:<20} | {tuple(feat.shape)}" )  # (C, H, W)

    # 시각화 top K
    show_topk_activation_grid( feat, f"Activation maps (Top-16) @ {title} ({layer_name})", topk=16 )

In [ ]:
### Convolution Kernel 시각화
    - Convolution filter 를 학습되면서 어떻게 변화되는지 시각화로 확인
    - conv1 은 3채널로 이미지처럼 볼 수 있음
    - 깊은 레이어는 입력 채널이 많아 평균/부분 채널만 보는 방식이 보편적

def show_conv_kernels( model, layer_name='conv1', max_kernels=64 ):
    layer = get_module_by_name( model, layer_name )

    w = layer.weight.detach().cpu()     # (0, I, kH, kW)
    O, I, kH, kW = w.shape
    K = min( 0, max_kernels )           # 시각화 커널 수 제한

    imgs = []
    for o in range(K):
        ker = w[o]
        if I == 3:  # RGB 입력이면 그대로 시각화
            img = (ker - ker.min()) / (ker.max() - ker.min + 1e-6)  # 0 ~ 1 정규화
            imgs.append( img )  # (3, kH, kW)
        else:       # 입력 채널이 많으면 평균내서 2D 로 시각화
            img2d = ker.mean(dim=0, keepdim=True)   # (1, kH, kW)
            img2d = (img2d - img2d.min()) / (img2d.max() - img2d.min() + 1e-6) # 0 ~ 1 정규화
            imgs.append( img2d ) # (1, kH, kW)

    imgs = torch.stack( imgs, dim=0)    # (K, C, kH, kW)
    
    grid = torchvision.utlis.make_grid( imgs, nrows=8, padding=2 )
    plt.figure( figsize=(12,6) )
    if grid.ndim == 3 and grid.shape[0] in (3, 4):  # CHW 이면 HWC 로 변환
        plt.imshow( grid.permute(1, 2, 0) )
    else:
        plt.imshow( grid.squeeze(0), cmap='gray')   # 1HW 이면 grayscale 로 변환
    plt.title(...)
    plt.axis('off')
    plt.show()


show_conv_kernels( model, 'conv1', max_kernels=64 )             # 첫번째 레이어 커널
show_conv_kernels( model, 'layer2.0.conv1', max_kernels=64)     # 중간 레이어 커널 (평균 시각화)
